# MoA — TabPFN (Transcriptome-based Cytotoxicity Prediction)

기존 다중 레이블(`tabpfn_basic_pca500_holdout.ipynb`) MoA 예측 파이프라인을 **세포 생존율(`c-`)기반 독성 이진 분류 모델**로 수정.

목적:
- 화학구조(SMILES)와 약물(drug_id) 매핑이 불가하다는 한계를 극복하기 위함
  -> 유전자 발현량(`g-`) 데이터의 변화 시그니처만으로 화합물의 잠재적인 세포 독성을 조기 예측

전처리 흐름: 
1. **정답지 생성** : `c-` 컬럼의 평균 세포 생존율을 계산하여, 특정 임계값(ex. -2.0)이하로 떨어지는 것을 고독성 / 아닌 것을 정상으로 라벨링
2. **Feature / Target 분리**
    : **`Target`** - 정답지 생성에 사용된 `c-` 컬럼들은 모델 입력(Feature)에서 완전히 제외(Drop)
    : **`Feature`** - 오직 `g-` 컬럼과 실험 조건(`cp_time`, `cp_dose`)만 남김.
3. **스케일링 &인코딩**
    - `cp_time`,`cp_dose` 정수 인코딩 / exposure_score 적용
    - `g-` : QuantileTransformer (train_split에 fit → val transform)
     >`QuantileTransformer` : 1등부터 순서대로 줄을 세우고, 0~1안의 수로 백분위로 표현
     >Train에만 fit하는 이유: TEST의 힌트를 얻을 수 있기에, 데이터 누수가 발생할 수 있음. 이에, Train에만 fit수행 (이후, transform하여 0~1사이 값으로 바꿈 -> Val transform)
     > Val transform: Train 데이터로 제작한 기준표를 기대로 가져와서 Test도 변경
4. **차원 축소**
    - PCA(n=500) — train_split 에 fit → 양쪽 transform
    - **`상호작용항 생성`** : 주성분 * exposure_score
5. **모델 학습**
    - 206개 target 각각에 TabPFNClassifier 독립 학습

In [2]:
# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 폰트
import platform

if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')

## 1. 경로 / 설정

In [3]:
import torch

DATA_DIR   = '../data/'
OUTPUT_DIR = 'moa_outputs_Cytotoxicity_Prediction'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu' # CUDA 지원 시 GPU 사용
SEED       = 42
N_SPLITS   = 5
VAL_FOLD   = 0

PCA_N_COMPONENTS = 500   # TabPFN v2 pretraining 피처 한계에 맞춤. 실험용 변수.
MAX_TARGETS      = None  # 디버그용. 전체는 None.

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"현재 사용 중인 장치: {DEVICE}")

현재 사용 중인 장치: cpu


In [2]:
#!pip install tabpfn iterative-stratification

In [4]:
import os

# .env 파일에서 직접 읽어 환경변수에 설정
env_path = os.path.join(os.getcwd(), ".env")
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith("TABPFN_TOKEN="):
                token = line.split("=", 1)[1]
                os.environ["TABPFN_TOKEN"] = token
                print(f"Token loaded: {token[:20]}...")
                break
else:
    print("WARNING: .env 파일을 찾을 수 없습니다!")

TABPFN_TOKEN = os.environ.get("TABPFN_TOKEN")


Token loaded: eyJhbGciOiJIUzI1NiIs...


In [5]:
# 이미지 저장 폴더
IMAGE_DIR = os.path.join(OUTPUT_DIR, 'images')

# 폴더가 없으면 새로 만들고, 이미 있으면 에러 없이 넘어갑니다.
os.makedirs(IMAGE_DIR, exist_ok=True) 
print(f"이미지 저장 폴더 생성: {IMAGE_DIR}")

이미지 저장 폴더 생성: moa_outputs_Cytotoxicity_Prediction\images


## 2. 데이터 로드 (train 만)

In [4]:
import numpy as np
import pandas as pd
import random, time, warnings
warnings.filterwarnings('ignore')

random.seed(SEED); np.random.seed(SEED)

# 데이터 로드
train_df = pd.read_csv(f'{DATA_DIR}train_features.csv')
test_df = pd.read_csv(f'{DATA_DIR}test_features.csv')

print('train_df (Original Features) :', train_df.shape)
print('test_df  (Original Features) :', test_df.shape)

train_df (Original Features) : (23814, 876)
test_df  (Original Features) : (3982, 876)


In [5]:
# Train_df와 Test_df에 drug_df의 drug_id를 병합 ['Sig_id'기준]
drug_df = pd.read_csv(f'{DATA_DIR}train_drug.csv')
train_df = pd.merge(train_df, drug_df[['drug_id', 'sig_id']], on='sig_id', how='left')
test_df = pd.merge(test_df, drug_df[['drug_id', 'sig_id']], on='sig_id', how='left')

print("Train_df와 Test_df에 drug_id 병합 완료")
print(f"Train_df shape: {train_df.shape}, Test_df shape: {test_df.shape}")

Train_df와 Test_df에 drug_id 병합 완료
Train_df shape: (23814, 877), Test_df shape: (3982, 877)


___
# **3. 전처리 시작**

## **1] 정답지 생성** 
- `c-` 컬럼의 평균 세포 생존율을 계산하여, 특정 임계값(ex. -2.0)이하로 떨어지는 것을 `고독성`, 아닌 것을 `정상`으로 라벨링

In [6]:
# 대조군(약물 넣지 않은 Sample) 제거
 # 약물의 독성 평가이므로, 대조군은 학습에서 제거.
train_df = train_df[train_df['cp_type'] != 'ctl_vehicle'].reset_index(drop=True)
test_df = test_df[test_df['cp_type'] != 'ctl_vehicle'].reset_index(drop=True)

print('train_df :', train_df.shape)
print('test_df  :', test_df.shape)

# 3. 'c-' 로 시작하는 모든 컬럼 이름 찾기
c_cols = [col for col in train_df.columns if col.startswith('c-')]
print(f" -> 성공적으로 찾아낸 c- 컬럼 개수: {len(c_cols)}개")

train_df['c_mean'] = train_df[c_cols].mean(axis=1)
print("\n=== 3. c_mean 데이터 분포 (정상 작동 확인) ===")
print(train_df['c_mean'].describe())

# 4. c-mean이 -2.0 이하면 독성(1), 아니면 정상(0) 라벨링
train_df['toxicity'] = (train_df['c_mean'] <= -2.0).astype(int)
print("\n=== 4. 독성 데이터(Label=1) 개수 확인 ===")
print(f" -> 생성된 독성 데이터 수: {train_df['toxicity'].sum()}개 / 전체 {len(train_df)}개")

train_df : (21948, 877)
test_df  : (3624, 877)
 -> 성공적으로 찾아낸 c- 컬럼 개수: 100개

=== 3. c_mean 데이터 분포 (정상 작동 확인) ===
count    21948.000000
mean        -0.483241
std          1.790391
min         -9.825370
25%         -0.302078
50%         -0.000830
75%          0.227714
max          2.851115
Name: c_mean, dtype: float64

=== 4. 독성 데이터(Label=1) 개수 확인 ===
 -> 생성된 독성 데이터 수: 1756개 / 전체 21948개


In [7]:
print("\n=== 기준별 독성 물질(Label=1) 개수 비교 ===")
threshold_names = ["Z-score -1.0", "Z-score -1.5", "Z-score -2.0", "하위 5%", "하위 10%"]

# 실제 계산에 사용할 기준점들
thresholds_to_test = [
    -1.0, 
    -1.5, 
    -2.0, 
    train_df['c_mean'].quantile(0.05), 
    train_df['c_mean'].quantile(0.10)
]
for name, th in zip(threshold_names, thresholds_to_test):
    # 해당 기준을 적용했을 때 독성 물질의 개수
    toxic_count = (train_df['c_mean'] <= th).sum()
    total_count = len(train_df)
    ratio = (toxic_count / total_count) * 100
    print(f"{name} (기준점: {th:6.3f}) -> 독성물질: {toxic_count:4d}개 ({ratio:.1f}%)")


=== 기준별 독성 물질(Label=1) 개수 비교 ===
Z-score -1.0 (기준점: -1.000) -> 독성물질: 2415개 (11.0%)
Z-score -1.5 (기준점: -1.500) -> 독성물질: 1990개 (9.1%)
Z-score -2.0 (기준점: -2.000) -> 독성물질: 1756개 (8.0%)
하위 5% (기준점: -4.477) -> 독성물질: 1098개 (5.0%)
하위 10% (기준점: -1.213) -> 독성물질: 2195개 (10.0%)


___
## **2] Feature / Target 분리 (데이터 누수 방지)**
- 데이터 누수: 동일 실험의 약물의 데이터를 무작위 분할할 경우! 특정 패턴만 보고, 독성이 있다고 판단할 수 있음
 -> 이에, drug_id를 가져와 같은 drug_id만 따로 추출해 확인하고자 함.

In [8]:
# 모델이 학습할 문제(X)와 정답(y) 나누기
cols_to_drop = c_cols + ['sig_id', 'c_mean', 'toxicity', 'cp_type', 'drug_id']

X = train_df.drop(columns=cols_to_drop, errors='ignore')
y = train_df['toxicity']                

# 데이터 누수 방지 -> drug_id 따로 추출
groups = train_df['drug_id']

# test도 동일하게 제작
X_test = test_df.drop(columns=c_cols + ['sig_id', 'cp_type'], errors='ignore')

print("Feature, Target 분리 완료")
print(f"\n문제지 크기: {X.shape}")
print(f"정답지 크기: {y.shape}")
print(f"테스트 문제지 크기: {X_test.shape}")

Feature, Target 분리 완료

문제지 크기: (21948, 774)
정답지 크기: (21948,)
테스트 문제지 크기: (3624, 775)


___
## **3] 스케일링 및 인코딩**
- `cp_time`,`cp_dose` 정수 인코딩
- `g-` : QuantileTransformer (train_split에 fit → val transform)

___
### **Train과 Test 데이터 나누기**
___

In [9]:
# Train, Test 데이터 생성
 # stratify=y 를 통해 '독성 물질(Label=1)'이 양쪽 그룹에 골고루 분포하도록 나눔
from sklearn.model_selection import train_test_split, GroupShuffleSplit

# 같은 drug_id 가진 데이터 함께 묶어 분할 수행
drug_s = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(drug_s.split(X, y, groups=groups))

print("Train, Test 데이터 나누기")
# .copy()로 독립 복사본 생성 → 인코딩 셀에서 새 컬럼 할당 시 SettingWithCopyWarning 방지
X_train = X.iloc[train_idx].copy()
X_val   = X.iloc[val_idx].copy()
y_train = y.iloc[train_idx]
y_val   = y.iloc[val_idx]

print("Train, Test 데이터 나누기 완료")
print(f"\n학습 데이터 크기: {X_train.shape}, 검증용 데이터 크기: {X_val.shape}")

Train, Test 데이터 나누기
Train, Test 데이터 나누기 완료

학습 데이터 크기: (17638, 774), 검증용 데이터 크기: (4310, 774)


___
### **스케일링 및 인코딩**

- 유전자와 실험 조건 데이터는 성격이 다르므로, 스케일링 및 PCA 차원 축소단계에서 다르게 다룸
___

In [10]:
# 노출 지수 생성(최소 1~최대 6)
for df in [X_train, X_val, X_test]:
    # 범주형 데이터(cp_time, cp_dose) 정수 인코딩
    # 시간(24h, 48h, 72h) -> 0, 1, 2
    # 용량(D1, D2) -> 0, 1
    df['cp_time_enc'] = df['cp_time'].map({24: 0, 48: 1, 72: 2})
    df['cp_dose_enc'] = df['cp_dose'].map({'D1': 0, 'D2': 1})
    
    # 노출 지수(exposure_score) 계산 가중치
    time_weight = df['cp_time'].map({24: 1, 48: 2, 72: 3})
    dose_weight = df['cp_dose'].map({'D1': 1, 'D2': 2})
    
    # 노출 지수 생성
    df['exposure_score'] = time_weight * dose_weight

# 유전자(g-)와 분리
g_cols = [col for col in X_train.columns if col.startswith('g-')]
# 모델에 입력할 추가 변수에 노출 지수 포함
other_cols = ['cp_time_enc', 'cp_dose_enc', 'exposure_score']

In [11]:
from sklearn.preprocessing import QuantileTransformer

# 스케일링 (QuantileTransformer)
qt = QuantileTransformer(output_distribution='normal', random_state=SEED)
X_train[g_cols] = qt.fit_transform(X_train[g_cols])
X_val[g_cols]   = qt.transform(X_val[g_cols])
X_test[g_cols]  = qt.transform(X_test[g_cols])

# Numpy 배열로 변환 (g- 컬럼만 사용 — cp_dose가 'D1'/'D2' 문자열이므로 전체 변환 불가)
X_train_raw = X_train[g_cols].astype(np.float32).values
y_train     = y_train.values.astype(np.float32)
X_val_raw   = X_val[g_cols].astype(np.float32).values
y_val       = y_val.values.astype(np.float32)

print("스케일링 완료")
print('X_train_raw:', X_train_raw.shape, ' Y_train:', y_train.shape)
print('X_val_raw  :', X_val_raw.shape,   ' Y_val  :', y_val.shape)


스케일링 완료
X_train_raw: (17638, 772)  Y_train: (17638,)
X_val_raw  : (4310, 772)  Y_val  : (4310,)


___
## **4] 차원 축소(PCA)**
- train_split 에 fit → 양쪽 transform

In [12]:
from sklearn.decomposition import PCA
# PCA 설정
n_comp = min(PCA_N_COMPONENTS, X_train_raw.shape[1], X_train_raw.shape[0])
pca = PCA(n_components=n_comp, random_state=SEED)
pca.fit(X_train_raw)

X_train_g = pca.fit_transform(X_train[g_cols]).astype(np.float32)
X_val_g   = pca.transform(X_val[g_cols]).astype(np.float32)
X_test_g  = pca.transform(X_test[g_cols]).astype(np.float32)

print(f'PCA n_components={n_comp}  explained_variance_ratio sum={pca.explained_variance_ratio_.sum():.4f}')
print('X_train_g:', X_train_g.shape, ' X_val_g:', X_val_g.shape, ' X_test_g:', X_test_g.shape)


# 상호직용하는 항 생성 
def create_interaction_terms(X_df, X_pca, n_interact=5):
    # 노출 지수를 가져옴
    exposure = X_df['exposure_score'].values.reshape(-1, 1)
    # PCA 상위 n_interact 개 특성과 노출 지수를 곱함
    interaction = X_pca[:, :n_interact] * exposure
    return interaction

train_interact = create_interaction_terms(X_train, X_train_g)
val_interact   = create_interaction_terms(X_val, X_val_g)
test_interact  = create_interaction_terms(X_test, X_test_g)

# 다른 변수(cp_time_enc, cp_dose_enc) + pca 유전자 +상호작용
X_train_final = np.hstack([X_train[other_cols].values, X_train_g, train_interact])
X_val_final   = np.hstack([X_val[other_cols].values, X_val_g, val_interact])
X_test_final  = np.hstack([X_test[other_cols].values, X_test_g, test_interact])

print("\n최종 데이터셋 준비 완료")
print(f"X_train_final: {X_train_final.shape}")
print(f"X_val_final  : {X_val_final.shape}")
print(f"X_test_final : {X_test_final.shape}")

PCA n_components=500  explained_variance_ratio sum=0.9135
X_train_g: (17638, 500)  X_val_g: (4310, 500)  X_test_g: (3624, 500)

최종 데이터셋 준비 완료
X_train_final: (17638, 508)
X_val_final  : (4310, 508)
X_test_final : (3624, 508)


___
## **5] 모델 학습** 
- TabPFN : 206개 binary classifier 루프 
           [`TabPFNClassifier`방법 활용]

In [13]:
from tabpfn import TabPFNClassifier
from sklearn.metrics import roc_auc_score, log_loss
import time # 모델 학습 및 평가 시간 측정
import numpy as np

# ==========================================
# 데이터 500개 샘플링
# SAMPLE_SIZE = 500
# X_train_final = X_train_final[:SAMPLE_SIZE]
# y_train = y_train[:SAMPLE_SIZE]
# X_val_final = X_val_final[:SAMPLE_SIZE]
# y_val = y_val[:SAMPLE_SIZE]
# X_val = X_val[:SAMPLE_SIZE]
# print(f"테스트용 데이터 세팅 완료! Train: {X_train_final.shape}, Val: {X_val_final.shape}")

In [14]:
# 1. 모델 생성
def make_tabpfn():
    return TabPFNClassifier(
        device=DEVICE,
        random_state=SEED,
        ignore_pretraining_limits=True,
    )

# 타겟 리스트 생성 (현재는 'toxicity' 하나지만, 확장 가능하도록 리스트로 유지)
target_cols = ['toxicity']

targets_to_run = target_cols if MAX_TARGETS is None else target_cols[:MAX_TARGETS]
print(f'Running TabPFN on {len(targets_to_run)} targets …')

Running TabPFN on 1 targets …


In [ ]:
# 2. 학습 및 예측 루프
pred_matrix = np.zeros((X_val_final.shape[0], len(targets_to_run)), dtype=np.float32)
meta = []
t0 = time.time()

for i, tgt in enumerate(targets_to_run):
    print(f"[{i+1}/{len(targets_to_run)}] {tgt:20s} 학습 시작...")
    
    # y_train이 1차원(9967,)인 경우를 위해 인덱싱 방식 변경
    if y_train.ndim == 1:
        y_curr = y_train.astype(int)
    else:
        j = target_cols.index(tgt)
        y_curr = y_train[:, j].astype(int)
        
    n_pos = int(y_curr.sum())

    if n_pos == 0:
        pred_matrix[:, i] = 0.0
        meta.append((tgt, n_pos, 0.0, 0.0))
        continue

    ts = time.time()
    clf = make_tabpfn()
    clf.fit(X_train_final, y_curr)
    proba = clf.predict_proba(X_val_final)
    
    # '독성 있음(1)'에 대한 확률값 추출
     # [:, 1] : '독성(1)'일 확률값 의미.
    pos_idx = list(clf.classes_).index(1)
    pred_matrix[:, i] = proba[:, pos_idx].astype(np.float32)
    dt = time.time() - ts
    meta.append((tgt, n_pos, float(pred_matrix[:, i].mean()), dt))

    if (i+1) % 10 == 0 or i == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i+1) * (len(targets_to_run) - (i+1))
        print(f'[{i+1:3d}/{len(targets_to_run)}] {tgt:40s} n_pos={n_pos:4d} dt={dt:5.1f}s  elapsed={elapsed/60:5.1f}m  ETA={eta/60:5.1f}m')

print(f'total: {(time.time()-t0)/60:.1f} min')

[1/1] toxicity             학습 시작...


In [ ]:
# 3. 검증 데이터 예측 (결과 확인용)
proba = clf.predict_proba(X_val_final)
s_idx = list(clf.classes_).index(1)
pred_matrix[:, i] = proba[:, s_idx].astype(np.float32)
    
# 4. 성능 평가: AUC 및 Log-Loss
auc_score = roc_auc_score(y_val, pred_matrix[:, 0])
loss_score = log_loss(y_val, proba)

print(f"\n {tgt} 학습 완료(소요시간: {time.time()-t0:.1f}s)")
print(f" -> AUC: {auc_score:.4f} | Log-Loss: {loss_score:.4f}")

## **4. Val 예측 저장**

In [ ]:
val_pred_df = pd.DataFrame(pred_matrix, columns=target_cols)
val_pred_df.insert(0, 'sig_id', train_df.loc[X_val.index, 'sig_id'].values)
pred_path = f'{OUTPUT_DIR}/val_pred_tabpfn_Cytotoxicity_prediction_split_pca{n_comp}.csv'
val_pred_df.to_csv(pred_path, index=False)
print('wrote', pred_path)
val_pred_df.head()

## **5. 평가 : mean column-wise log-loss**

In [ ]:
y_true_arr = y_val.astype(np.float32)
y_pred_arr = np.clip(pred_matrix.astype(np.float32), 1e-15, 1 - 1e-15)

loss = -(y_true_arr * np.log(y_pred_arr) + (1 - y_true_arr) * np.log(1 - y_pred_arr)).mean()
print(f'Mean column-wise log-loss (TabPFN drug_id split + PCA{n_comp}, fold={VAL_FOLD}): {loss:.5f}')

meta_df = pd.DataFrame(meta, columns=['target', 'n_pos', 'pred_mean', 'fit_predict_sec'])
meta_df.to_csv(f'{OUTPUT_DIR}/tabpfn_drug_id_split_pca{n_comp}_per_target.csv', index=False)
meta_df.head()

___
### **결과 해석**
- **TabPFN과 Logistic Regression 비교**

    : 평균 독성 확률이 **배 높은 것을 통해 더 민감하게 잡아내고 있음을 확인
   0과 1 양극단으로 갈리지만, 중앙값이 더 높고, 분포의 꼬리가 넓음

   => 이는 애매한 영역(일명, 잠재적 독성)을 TabPFN이 더 잘 잡아내고 있음을 확인.
___
### **한계점 및 해결 방안**
- 연산 시간이 오래 걸림
 1) `AutoEncoder`을 통한 차원 축소를 거쳐 연산 시간을 줄인다.
 2) `앙상블`을 통해 로지스틱 회귀의 `toxicity_proba`의 값과 TabPFN의 `toxicity`의 값을 각각 `0.5`의 비율로 더해 평균을 내보는 방법도 있다.

## **6. 시각화**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 데이터 준비
analysis_df = X_val.copy()
analysis_df['toxicity_proba'] = pred_matrix 
analysis_df['true_label'] = y_val

# 노출 지수(exposure_score) 별 평균 예측 확률 계산 후 데이터프레임으로 변환
grouped_data = analysis_df.groupby('exposure_score')['toxicity_proba'].mean().reset_index()

# 2. 시각화
plt.figure(figsize=(10, 6))

# 막대 그래프 (Bar plot) 그리기
ax = sns.barplot(
    x='exposure_score', 
    y='toxicity_proba', 
    data=grouped_data, 
    palette='Reds' 
)
x_coords = np.arange(len(grouped_data))
# 꺾은선(추세선) 추가
sns.lineplot(
    x=x_coords,
    y='toxicity_proba', 
    data=grouped_data, 
    color='black', 
    marker='o', 
    linewidth=2,
    markersize=8,
    ax=ax # 막대 그래프 위에 겹쳐 그림.
)

# 제목 및 라벨링
plt.title('Exposure-Dependent Toxicity Sensitivity', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Exposure Score (Dose x Time)', fontsize=13)
plt.ylabel('Mean Predicted Toxicity Probability', fontsize=13)

# 출력
plt.tight_layout()
plt.show()